# Task 3: Domain generalization on PACS
Sources: Photo, Art Painting, Cartoon. Target: Sketch

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import copy
import json
import math
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torchvision.models import ResNet18_Weights, resnet18

CFG = {
    # protocol shared with task 2 (identical values)
    "seed": 6304,
    "sources": ["photo", "art_painting", "cartoon"],
    "target": "sketch",
    "resize": 256, "crop": 224,
    "per_source_batch": 8,           # 3 x 8 = 24 source images per update, nothing else
    "max_epochs": 30, "patience": 5,
    "lr": 1e-4, "weight_decay": 1e-4,
    "mmd_bandwidth_mults": [0.5, 1.0, 2.0],

    # task 3 methods
    "lambda_dg": 1.0,                # DAN-DG main comparison
    "sam_rho": 0.05,                 # SAM main comparison
    "study_lambdas": [0.1, 1.0, 10.0],

    # source-side diagnostics
    "separability": {"test_size": 0.3, "C": 1.0},
    "sharpness": {"per_source": 32, "radius": 0.05},
}
RUNS = {
    "erm":           {"method": "erm"},                       # loaded from task 2, never retrained
    "dan_dg":        {"method": "dan_dg", "lam": 1.0},
    "sam":           {"method": "sam", "rho": 0.05},
    "dan_dg_lam0.1": {"method": "dan_dg", "lam": 0.1},
    "dan_dg_lam10":  {"method": "dan_dg", "lam": 10.0},
}
MAIN_RUNS = ["erm", "dan_dg", "sam"]
STUDY_RUNS = ["dan_dg_lam0.1", "dan_dg", "dan_dg_lam10"]
RUN_LABELS = {"erm": "ERM", "dan_dg": "DAN-DG", "sam": "SAM",
              "dan_dg_lam0.1": "DAN-DG (lambda=0.1)", "dan_dg_lam10": "DAN-DG (lambda=10)"}

REPO = Path("/content/drive/MyDrive/atml-pa1")
RESULTS = REPO / "task3" / "results"
FIGS = RESULTS / "figures"
LOGS = RESULTS / "logs"
TASK2_RESULTS = REPO / "task2" / "results"
SPLIT_FILE = REPO / "shared" / "splits" / "pacs_sketch_seed6304.json"
CACHE = Path("/content/drive/MyDrive/pa1-data/pacs")     # same tensor and checkpoint folder as task 2
CKPT = CACHE / "checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [3]:
if not REPO.exists():
    !git clone https://github.com/mardyweb/atml-pa1.git {REPO}

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)


def save_table(df, name):
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    try:
        df.to_latex(RESULTS / f"{name}.tex", index=False, float_format="%.3f", escape=True)
    except Exception:
        pass
    display(df.round(3))


for folder in [RESULTS, FIGS, LOGS]:
    folder.mkdir(parents=True, exist_ok=True)
for needed in [CACHE / "pacs_256.pt", CKPT / "source_only.pt", SPLIT_FILE, TASK2_RESULTS / "logs" / "source_only.json"]:
    assert needed.exists(), f"task 2 output missing: {needed}"
set_seed(CFG["seed"])
save_json({"config": CFG, "runs": RUNS, "device": str(DEVICE),
           "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                        "sklearn": sklearn.__version__, "numpy": np.__version__}}, RESULTS / "run_info.json")

# ---- same look as the other tasks ----
RUN_COLORS = {"erm": "#2E4057", "dan_dg": "#00798C", "sam": "#D1495B",
              "dan_dg_lam0.1": "#7FBFC4", "dan_dg_lam10": "#003F4A", "dan_task2": "#EDAE49"}
RUN_MARKERS = {"erm": "o", "dan_dg": "s", "sam": "D", "dan_dg_lam0.1": "v", "dan_dg_lam10": "P", "dan_task2": "^"}
INK = "#2B2B2B"
GOOD, BAD = "#00798C", "#D1495B"
TEAL_CMAP = LinearSegmentedColormap.from_list("teal", ["#FBF6EC", "#BFE0DA", "#4FA6A6", "#00798C", "#003F4A"])
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 8.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E4E0D6", "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.dpi": 110, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save_fig(fig, name):
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

## Data: the three source domains only

In [5]:
# the cached tensor holds all four domains. keep the source rows only and drop the rest right away, so nothing below this cell can reach a Sketch image or label.
pacs = torch.load(CACHE / "pacs_256.pt")
domain_all = np.array(pacs["domains"])
CLASS_NAMES = pacs["class_names"]
N_CLASSES = len(CLASS_NAMES)
SOURCES = CFG["sources"]

src_mask = domain_all != CFG["target"]
IMAGES = pacs["images"][torch.from_numpy(src_mask)]          # (N_source, 3, 256, 256) uint8
LABELS = pacs["labels"][torch.from_numpy(src_mask)]
DOMAIN_OF = domain_all[src_mask]
del pacs

# task 2 split indices refer to rows of the full tensor; translate them to rows of the source tensor
to_local = np.cumsum(src_mask) - 1
SPLITS = {d: {k: sorted(int(to_local[i]) for i in v) for k, v in s.items()}
          for d, s in load_json(SPLIT_FILE)["splits"].items()}
for d in SOURCES:
    assert all(DOMAIN_OF[i] == d for i in SPLITS[d]["train"] + SPLITS[d]["val"])

STEPS_PER_EPOCH = math.ceil(max(len(SPLITS[d]["train"]) for d in SOURCES) / CFG["per_source_batch"])
print({d: (len(s["train"]), len(s["val"])) for d, s in SPLITS.items()}, "| updates per epoch:", STEPS_PER_EPOCH)
print("domains in memory:", sorted(set(DOMAIN_OF)))

{'photo': (1336, 334), 'art_painting': (1638, 410), 'cartoon': (1875, 469)} | updates per epoch: 235
domains in memory: [np.str_('art_painting'), np.str_('cartoon'), np.str_('photo')]


## Model, MMD, sampling (identical to task 2)

In [6]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


class Net(nn.Module):
    """ImageNet ResNet-18 with a new 7-class linear head. Returns the 512-d feature and the logits."""

    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Identity()
        self.head = nn.Linear(512, N_CLASSES)

    def forward(self, x_uint8):
        x = (x_uint8.float() / 255 - MEAN) / STD
        feat = self.backbone(x)
        return feat, self.head(feat)


def train_mode(model):
    """Training mode, but BatchNorm keeps its ImageNet running mean/var (scale and shift still train)."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


def mmd2(feat_a, feat_b):
    """Squared MMD with a sum of three RBF kernels, bandwidths 0.5, 1 and 2 times the median
    pairwise squared distance of the combined batch of the two inputs (same code as task 2)"""
    z = torch.cat([feat_a, feat_b]).float()
    sq = (z * z).sum(dim=1)
    d2 = (sq[:, None] + sq[None, :] - 2 * z @ z.T).clamp_min(0)
    off_diag = ~torch.eye(len(z), dtype=torch.bool, device=z.device)
    median = d2.detach()[off_diag].median().clamp_min(1e-8)
    k = sum(torch.exp(-d2 / (m * median)) for m in CFG["mmd_bandwidth_mults"])
    n = len(feat_a)
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()


class Cycler:
    """Endless shuffled mini-batches from a fixed set of image indices."""

    def __init__(self, indices, batch_size, seed):
        self.indices, self.batch_size = np.asarray(indices), batch_size
        self.rng = np.random.default_rng(seed)
        self.queue = []

    def next(self):
        while len(self.queue) < self.batch_size:
            self.queue += self.rng.permutation(self.indices).tolist()
        batch, self.queue = self.queue[:self.batch_size], self.queue[self.batch_size:]
        return batch


def augmented_batch(indices, rng):
    """Random 224 crop + horizontal flip on the stored 256x256 tensors."""
    crop, room = CFG["crop"], CFG["resize"] - CFG["crop"]
    out = torch.empty((len(indices), 3, crop, crop), dtype=torch.uint8)
    for k, i in enumerate(indices):
        top, left = rng.integers(0, room + 1, size=2)
        img = IMAGES[i, :, top:top + crop, left:left + crop]
        out[k] = img.flip(-1) if rng.random() < 0.5 else img
    return out.to(DEVICE, non_blocking=True)


def center_crop(indices, images=None):
    images = IMAGES if images is None else images
    off = (CFG["resize"] - CFG["crop"]) // 2
    return images[torch.as_tensor(indices)][:, :, off:off + CFG["crop"], off:off + CFG["crop"]]


@torch.no_grad()
def forward_all(model, indices, images=None, batch_size=256):
    """Features and predictions for a list of images (center crop, eval mode)."""
    model.eval()
    feats, preds = [], []
    for start in range(0, len(indices), batch_size):
        f, logits = model(center_crop(indices[start:start + batch_size], images).to(DEVICE))
        feats.append(f.float().cpu())
        preds.append(logits.argmax(1).cpu())
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


def source_validation(model):
    """Accuracy and macro-F1 on each source validation split, plus mean and worst."""
    out = {}
    for d in SOURCES:
        idx = SPLITS[d]["val"]
        _, pred = forward_all(model, idx)
        y = LABELS[idx].numpy()
        out[d] = {"acc": float((pred == y).mean()),
                  "f1": float(f1_score(y, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))}
    out["mean_acc"] = float(np.mean([out[d]["acc"] for d in SOURCES]))
    out["mean_f1"] = float(np.mean([out[d]["f1"] for d in SOURCES]))
    out["worst_acc"] = float(min(out[d]["acc"] for d in SOURCES))
    out["worst_f1"] = float(min(out[d]["f1"] for d in SOURCES))
    return out

## Training loop for DAN-DG and SAM
Same initialization, source batches, crops and flips as the task 2 Source-only run.

In [7]:
def sam_step(model, opt, loss_fn, rho):
    """One SAM update: ascend to the worst point within radius rho, then step with that gradient."""
    loss = loss_fn()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    params = [p for p in model.parameters() if p.grad is not None]
    grad_norm = torch.norm(torch.stack([p.grad.norm(2) for p in params]), 2)
    scale = rho / (grad_norm + 1e-12)
    with torch.no_grad():
        saved = [p.detach().clone() for p in params]
        for p in params:
            p.add_(p.grad * scale)          # theta + epsilon
    loss_adv = loss_fn()                    # second forward/backward at the perturbed point
    opt.zero_grad(set_to_none=True)
    loss_adv.backward()
    with torch.no_grad():
        for p, s0 in zip(params, saved):
            p.copy_(s0)                     # exactly back to theta, keeping the perturbed gradient
    opt.step()
    return loss.item(), loss_adv.item()


def train_run(run_name):
    spec = RUNS[run_name]
    assert spec["method"] != "erm", "ERM is the task 2 Source-only checkpoint and is not retrained"
    ckpt_path, log_path = CKPT / f"task3_{run_name}.pt", LOGS / f"{run_name}.json"
    if ckpt_path.exists() and log_path.exists():
        print(f"{run_name}: already trained, skipping")
        return

    set_seed(CFG["seed"])
    model = Net().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    # the same random streams as task 2, so every method sees the same source batches
    seed = CFG["seed"]
    src_cyclers = [Cycler(SPLITS[d]["train"], CFG["per_source_batch"], [seed, k]) for k, d in enumerate(SOURCES)]
    src_aug = np.random.default_rng([seed, 100])
    n_per, n_dom = CFG["per_source_batch"], len(SOURCES)
    pairs = [(a, b) for a in range(n_dom) for b in range(a + 1, n_dom)]

    best_f1, best_state, best_epoch, bad_epochs, log = -1.0, None, 0, 0, []
    for epoch in range(1, CFG["max_epochs"] + 1):
        t0 = time.time()
        train_mode(model)                    # BatchNorm statistics stay frozen in every pass
        sums = {"cls": 0.0, "mmd": 0.0, "sam_adv": 0.0}

        for _ in range(STEPS_PER_EPOCH):
            src_idx = sum([c.next() for c in src_cyclers], [])
            ys = LABELS[src_idx].to(DEVICE)
            xs = augmented_batch(src_idx, src_aug)

            if spec["method"] == "dan_dg":
                feat, logits = model(xs)
                cls_loss = F.cross_entropy(logits, ys)
                # average MMD over the three unordered pairs of source domains
                blocks = [feat[k * n_per:(k + 1) * n_per] for k in range(n_dom)]
                mmd_loss = sum(mmd2(blocks[a], blocks[b]) for a, b in pairs) / len(pairs)
                loss = cls_loss + spec["lam"] * mmd_loss
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                sums["cls"] += cls_loss.item()
                sums["mmd"] += mmd_loss.item()
            else:   # sam
                loss_fn = lambda: F.cross_entropy(model(xs)[1], ys)
                cls_loss, adv_loss = sam_step(model, opt, loss_fn, spec["rho"])
                sums["cls"] += cls_loss
                sums["sam_adv"] += adv_loss

        # model selection on the source validation splits only
        val = source_validation(model)
        entry = {"epoch": epoch, "cls_loss": sums["cls"] / STEPS_PER_EPOCH,
                 "mmd_loss": sums["mmd"] / STEPS_PER_EPOCH if spec["method"] == "dan_dg" else None,
                 "sam_perturbed_loss": sums["sam_adv"] / STEPS_PER_EPOCH if spec["method"] == "sam" else None,
                 "val": val}
        log.append(entry)
        extra = f"mmd {entry['mmd_loss']:.4f}" if spec["method"] == "dan_dg" else f"perturbed {entry['sam_perturbed_loss']:.3f}"
        print(f"{run_name} | epoch {epoch:2d} | cls {entry['cls_loss']:.3f} | {extra} | "
              f"val macro-F1 {val['mean_f1']:.4f} (worst {val['worst_f1']:.4f}) | {time.time() - t0:.0f}s")

        if val["mean_f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = val["mean_f1"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= CFG["patience"]:
                break

    torch.save({"model": best_state, "best_epoch": best_epoch, "best_mean_val_f1": best_f1, "spec": spec}, ckpt_path)
    save_json({"run": run_name, "spec": spec, "best_epoch": best_epoch, "best_mean_val_f1": best_f1,
               "epochs_run": len(log), "steps_per_epoch": STEPS_PER_EPOCH, "log": log}, log_path)
    print(f"{run_name}: kept epoch {best_epoch} (mean source-val macro-F1 {best_f1:.4f})")

## Step 1: ERM baseline (the task 2 Source-only checkpoint)

In [8]:
# copy its training log next to the task 3 logs so the curves can be drawn together
shutil.copy(TASK2_RESULTS / "logs" / "source_only.json", LOGS / "erm.json")
erm_log = load_json(LOGS / "erm.json")
print(f"ERM: task 2 checkpoint from epoch {erm_log['best_epoch']} (mean source-val macro-F1 {erm_log['best_mean_val_f1']:.4f})")

ERM: task 2 checkpoint from epoch 13 (mean source-val macro-F1 0.9542)


## Step 2: DAN-DG (pairwise source-domain MMD, lambda = 1)

In [9]:
train_run("dan_dg")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]


dan_dg | epoch  1 | cls 1.942 | mmd 0.5732 | val macro-F1 0.0507 (worst 0.0421) | 23s
dan_dg | epoch  2 | cls 1.936 | mmd 0.5363 | val macro-F1 0.0507 (worst 0.0421) | 22s
dan_dg | epoch  3 | cls 1.934 | mmd 0.5181 | val macro-F1 0.0507 (worst 0.0421) | 22s
dan_dg | epoch  4 | cls 1.938 | mmd 0.1813 | val macro-F1 0.0507 (worst 0.0421) | 23s
dan_dg | epoch  5 | cls 1.936 | mmd 0.0155 | val macro-F1 0.0507 (worst 0.0421) | 24s
dan_dg | epoch  6 | cls 1.934 | mmd 0.0203 | val macro-F1 0.0507 (worst 0.0421) | 23s
dan_dg: kept epoch 1 (mean source-val macro-F1 0.0507)


## Step 3: SAM (rho = 0.05)

In [10]:
train_run("sam")

sam | epoch  1 | cls 0.723 | perturbed 1.044 | val macro-F1 0.8510 (worst 0.8262) | 45s
sam | epoch  2 | cls 0.240 | perturbed 0.493 | val macro-F1 0.9332 (worst 0.9037) | 44s
sam | epoch  3 | cls 0.160 | perturbed 0.365 | val macro-F1 0.9386 (worst 0.8991) | 45s
sam | epoch  4 | cls 0.114 | perturbed 0.299 | val macro-F1 0.9390 (worst 0.9155) | 45s
sam | epoch  5 | cls 0.078 | perturbed 0.231 | val macro-F1 0.9523 (worst 0.9215) | 45s
sam | epoch  6 | cls 0.059 | perturbed 0.208 | val macro-F1 0.9476 (worst 0.9164) | 45s
sam | epoch  7 | cls 0.046 | perturbed 0.182 | val macro-F1 0.9535 (worst 0.9389) | 45s
sam | epoch  8 | cls 0.038 | perturbed 0.162 | val macro-F1 0.9454 (worst 0.9131) | 44s
sam | epoch  9 | cls 0.026 | perturbed 0.137 | val macro-F1 0.9565 (worst 0.9417) | 44s
sam | epoch 10 | cls 0.024 | perturbed 0.138 | val macro-F1 0.9492 (worst 0.9237) | 44s
sam | epoch 11 | cls 0.020 | perturbed 0.121 | val macro-F1 0.9366 (worst 0.8850) | 45s
sam | epoch 12 | cls 0.019 | per

## Step 5: Controlled study, DAN-DG with lambda = 0.1 and 10
Trained now so that every checkpoint exists before any diagnostic or Sketch evaluation.

In [11]:
train_run("dan_dg_lam0.1")
train_run("dan_dg_lam10")

dan_dg_lam0.1 | epoch  1 | cls 0.475 | mmd 0.5790 | val macro-F1 0.9032 (worst 0.8841) | 24s
dan_dg_lam0.1 | epoch  2 | cls 0.180 | mmd 0.5235 | val macro-F1 0.9386 (worst 0.9043) | 24s
dan_dg_lam0.1 | epoch  3 | cls 0.130 | mmd 0.4978 | val macro-F1 0.9218 (worst 0.8796) | 24s
dan_dg_lam0.1 | epoch  4 | cls 0.098 | mmd 0.4824 | val macro-F1 0.9322 (worst 0.9044) | 24s
dan_dg_lam0.1 | epoch  5 | cls 0.064 | mmd 0.4774 | val macro-F1 0.9511 (worst 0.9209) | 24s
dan_dg_lam0.1 | epoch  6 | cls 0.043 | mmd 0.4771 | val macro-F1 0.9405 (worst 0.9104) | 24s
dan_dg_lam0.1 | epoch  7 | cls 0.028 | mmd 0.4719 | val macro-F1 0.9389 (worst 0.9216) | 24s
dan_dg_lam0.1 | epoch  8 | cls 0.040 | mmd 0.4634 | val macro-F1 0.9321 (worst 0.9010) | 24s
dan_dg_lam0.1 | epoch  9 | cls 0.068 | mmd 0.4949 | val macro-F1 0.9380 (worst 0.9020) | 24s
dan_dg_lam0.1 | epoch 10 | cls 0.020 | mmd 0.4752 | val macro-F1 0.9456 (worst 0.9149) | 24s
dan_dg_lam0.1: kept epoch 5 (mean source-val macro-F1 0.9511)
dan_dg_l

## Step 4a: Source-side diagnostics (no Sketch involved)

In [12]:
def load_model(run):
    path = CKPT / "source_only.pt" if run == "erm" else CKPT / f"task3_{run}.pt"
    model = Net().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE)["model"])
    return model.eval()


def source_domain_separability(model):
    """Can a linear probe tell Photo, Art and Cartoon apart from the frozen features? 33.3% = it cannot."""
    sc, seed = CFG["separability"], CFG["seed"]
    rng = np.random.default_rng(seed)
    n = min(len(SPLITS[d]["val"]) for d in SOURCES)              # same number of features per domain
    feats, doms = [], []
    for k, d in enumerate(SOURCES):
        idx = rng.choice(SPLITS[d]["val"], size=n, replace=False)
        feats.append(forward_all(model, sorted(idx.tolist()))[0])
        doms.append(np.full(n, k))
    x, y = np.concatenate(feats), np.concatenate(doms)
    x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=sc["test_size"], stratify=y, random_state=seed)
    probe = LogisticRegression(C=sc["C"], max_iter=5000).fit(x_tr, y_tr)   # multinomial by default
    return float(probe.score(x_te, y_te))


# one fixed validation batch, 32 examples per source, shared by all models
_rng = np.random.default_rng(CFG["seed"])
SHARP_IDX = sum([sorted(_rng.choice(SPLITS[d]["val"], size=CFG["sharpness"]["per_source"], replace=False).tolist())
                 for d in SOURCES], [])
SHARP_X, SHARP_Y = center_crop(SHARP_IDX).to(DEVICE), LABELS[SHARP_IDX].to(DEVICE)


def sharpness_proxy(model):
    """Loss increase after one normalized gradient-ascent step of radius 0.05, in evaluation mode."""
    model.eval()
    params = [p for p in model.parameters()]
    model.zero_grad(set_to_none=True)
    base = F.cross_entropy(model(SHARP_X)[1], SHARP_Y)
    base.backward()
    grads = [p.grad if p.grad is not None else torch.zeros_like(p) for p in params]
    norm = torch.norm(torch.stack([g.norm(2) for g in grads]), 2)
    with torch.no_grad():
        saved = [p.detach().clone() for p in params]
        for p, g in zip(params, grads):
            p.add_(CFG["sharpness"]["radius"] * g / (norm + 1e-12))
        perturbed = F.cross_entropy(model(SHARP_X)[1], SHARP_Y)
        for p, s0 in zip(params, saved):
            p.copy_(s0)                     # exactly back to theta
    model.zero_grad(set_to_none=True)
    return {"loss": base.item(), "perturbed_loss": perturbed.item(), "delta_sharp": perturbed.item() - base.item(),
            "grad_norm": norm.item()}


DIAG = {}
for run in RUNS:
    model = load_model(run)
    DIAG[run] = {"val": source_validation(model),
                 "source_separability": source_domain_separability(model),
                 "sharpness": sharpness_proxy(model)}
    print(f"{run:14s} mean F1 {DIAG[run]['val']['mean_f1']:.4f} | worst F1 {DIAG[run]['val']['worst_f1']:.4f} | "
          f"separability {DIAG[run]['source_separability']:.3f} | sharpness {DIAG[run]['sharpness']['delta_sharp']:.4f}")
    del model
save_json({"sharpness_batch_indices_local": SHARP_IDX, "diagnostics": DIAG}, RESULTS / "source_diagnostics.json")

erm            mean F1 0.9542 | worst F1 0.9302 | separability 0.850 | sharpness 0.3062
dan_dg         mean F1 0.0507 | worst F1 0.0421 | separability 0.422 | sharpness 0.0109
sam            mean F1 0.9565 | worst F1 0.9417 | separability 0.857 | sharpness 0.1023
dan_dg_lam0.1  mean F1 0.9511 | worst F1 0.9209 | separability 0.854 | sharpness 0.1880
dan_dg_lam10   mean F1 0.0507 | worst F1 0.0421 | separability 0.332 | sharpness 0.0076


## Step 4b: Final evaluation on Sketch
Everything above is now fixed. This is the first and only place Sketch is read.

In [ ]:
missing = [r for r in RUNS if r != "erm" and not (CKPT / f"task3_{r}.pt").exists()]
assert not missing, f"train these first: {missing}"
save_json({"locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
           "selection_rule": "best mean macro-F1 over the three source validation splits",
           "checkpoints": {r: ({"best_epoch": erm_log["best_epoch"], "best_mean_val_f1": erm_log["best_mean_val_f1"]} if r == "erm"
                               else {k: load_json(LOGS / f"{r}.json")[k] for k in ("best_epoch", "best_mean_val_f1")}) for r in RUNS}},
          RESULTS / "locked_before_sketch_eval.json")

# now read the Sketch images and labels
pacs = torch.load(CACHE / "pacs_256.pt")
sk_mask = torch.from_numpy(np.array(pacs["domains"]) == CFG["target"])
SKETCH_IMAGES, SKETCH_LABELS = pacs["images"][sk_mask], pacs["labels"][sk_mask].numpy()
del pacs
print("sketch images:", len(SKETCH_IMAGES))

EVAL = {}
for run in RUNS:
    model = load_model(run)
    _, pred = forward_all(model, np.arange(len(SKETCH_IMAGES)), images=SKETCH_IMAGES)
    cm = confusion_matrix(SKETCH_LABELS, pred, labels=list(range(N_CLASSES)))
    EVAL[run] = {**DIAG[run],
                 "sketch_acc": float((pred == SKETCH_LABELS).mean()),
                 "sketch_f1": float(f1_score(SKETCH_LABELS, pred, average="macro")),
                 "per_class_acc": (cm.diagonal() / cm.sum(1)).tolist(),
                 "confusion": cm.tolist(), "sketch_pred": pred.tolist()}
    del model
save_json(EVAL, RESULTS / "final_evaluation.json")


def result_row(run):
    e, base = EVAL[run], EVAL["erm"]
    row = {"method": RUN_LABELS[run]}
    for d in SOURCES:
        row[f"{d} acc"], row[f"{d} F1"] = e["val"][d]["acc"], e["val"][d]["f1"]
    row.update({"mean src acc": e["val"]["mean_acc"], "mean src F1": e["val"]["mean_f1"],
                "worst src acc": e["val"]["worst_acc"], "worst src F1": e["val"]["worst_f1"],
                "sketch acc": e["sketch_acc"], "sketch F1": e["sketch_f1"],
                "sketch acc change": e["sketch_acc"] - base["sketch_acc"],
                "source sep.": e["source_separability"], "sharpness": e["sharpness"]["delta_sharp"]})
    return row


MAIN_TABLE = pd.DataFrame([result_row(r) for r in MAIN_RUNS])
save_table(MAIN_TABLE, "table_main_comparison")

sketch images: 3929


In [ ]:
# per-class Sketch accuracy, with the task 2 target-aware DAN next to the target-free DAN-DG
T2 = load_json(TASK2_RESULTS / "final_evaluation.json")
assert abs(T2["source_only"]["target_acc"] - EVAL["erm"]["sketch_acc"]) < 1e-6, "ERM must be the same model as task 2 Source-only"
rows = []
for c, name in enumerate(CLASS_NAMES):
    row = {"class": name, "n_sketch": int(np.sum(SKETCH_LABELS == c)), "ERM": EVAL["erm"]["per_class_acc"][c],
           "DAN (task 2)": T2["dan"]["per_class_acc"][c], "DAN-DG": EVAL["dan_dg"]["per_class_acc"][c],
           "SAM": EVAL["sam"]["per_class_acc"][c]}
    for k in ["DAN (task 2)", "DAN-DG", "SAM"]:
        row[f"{k} change"] = row[k] - row["ERM"]
    rows.append(row)
PER_CLASS = pd.DataFrame(rows)
save_table(PER_CLASS, "table_per_class_sketch")

# dominant confusion of every class for each task 3 model
rows = []
for run in MAIN_RUNS:
    cm = np.array(EVAL[run]["confusion"])
    for c, name in enumerate(CLASS_NAMES):
        wrong = cm[c].copy()
        wrong[c] = 0
        rows.append({"method": RUN_LABELS[run], "true class": name, "accuracy": cm[c, c] / cm[c].sum(),
                     "most confused with": CLASS_NAMES[int(wrong.argmax())] if wrong.max() > 0 else "nothing",
                     "share of class": wrong.max() / cm[c].sum()})
CONFUSIONS = pd.DataFrame(rows)
CONFUSIONS.to_csv(RESULTS / "table_dominant_confusions.csv", index=False)
for run in MAIN_RUNS[1:]:
    change = PER_CLASS[f"{RUN_LABELS[run]} change"].values
    for tag, c in [("largest gain", int(change.argmax())), ("largest drop", int(change.argmin()))]:
        before = CONFUSIONS[(CONFUSIONS.method == "ERM") & (CONFUSIONS["true class"] == CLASS_NAMES[c])].iloc[0]
        after = CONFUSIONS[(CONFUSIONS.method == RUN_LABELS[run]) & (CONFUSIONS["true class"] == CLASS_NAMES[c])].iloc[0]
        print(f"{RUN_LABELS[run]:6s} {tag}: {CLASS_NAMES[c]:9s} {100 * change[c]:+5.1f} pp | "
              f"ERM confuses it with {before['most confused with']} ({100 * before['share of class']:.0f}%), "
              f"{RUN_LABELS[run]} with {after['most confused with']} ({100 * after['share of class']:.0f}%)")

# target-aware vs target-free alignment, same ERM start, same MMD code
rows = [{"model": "ERM (shared baseline)", "sees Sketch images": "no",
         "mean src F1": EVAL["erm"]["val"]["mean_f1"], "sketch acc": EVAL["erm"]["sketch_acc"], "sketch F1": EVAL["erm"]["sketch_f1"]},
        {"model": "DAN, task 2 (lambda=1)", "sees Sketch images": "yes, unlabeled",
         "mean src F1": T2["dan"]["val"]["mean_f1"], "sketch acc": T2["dan"]["target_acc"], "sketch F1": T2["dan"]["target_f1"]},
        {"model": "DAN-DG, task 3 (lambda=1)", "sees Sketch images": "no",
         "mean src F1": EVAL["dan_dg"]["val"]["mean_f1"], "sketch acc": EVAL["dan_dg"]["sketch_acc"], "sketch F1": EVAL["dan_dg"]["sketch_f1"]}]
save_table(pd.DataFrame(rows), "table_dan_vs_dan_dg")

In [ ]:
# controlled study: alignment strength of DAN-DG (ERM = no alignment, for reference)
rows = []
for run, lam in [("erm", 0.0)] + list(zip(STUDY_RUNS, CFG["study_lambdas"])):
    e = EVAL[run]
    rows.append({"lambda": lam, "mean src F1": e["val"]["mean_f1"], "worst src F1": e["val"]["worst_f1"],
                 "source sep.": e["source_separability"], "sharpness": e["sharpness"]["delta_sharp"],
                 "sketch acc": e["sketch_acc"], "sketch F1": e["sketch_f1"],
                 "best epoch": erm_log["best_epoch"] if run == "erm" else load_json(LOGS / f"{run}.json")["best_epoch"]})
STUDY_TABLE = pd.DataFrame(rows)
save_table(STUDY_TABLE, "table_lambda_study")

## Figures

In [ ]:
# training curves: classification loss, MMD penalty, SAM's perturbed loss, source validation macro-F1
logs = {r: (erm_log if r == "erm" else load_json(LOGS / f"{r}.json")) for r in RUNS}
fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.05))


def curve(ax, run, key, **kw):
    ep = [e["epoch"] for e in logs[run]["log"]]
    vals = [e["val"]["mean_f1"] if key == "val_f1" else e[key] for e in logs[run]["log"]]
    ax.plot(ep, vals, color=RUN_COLORS[run], marker=RUN_MARKERS[run], markersize=3, linewidth=1.4, label=RUN_LABELS[run], **kw)


for run in MAIN_RUNS:
    curve(axes[0], run, "cls_loss")
    curve(axes[3], run, "val_f1")
for run in STUDY_RUNS:
    curve(axes[1], run, "mmd_loss")
curve(axes[2], "sam", "cls_loss")
curve(axes[2], "sam", "sam_perturbed_loss", linestyle=":")
for ax, title in zip(axes, ["source class. loss", "DAN-DG: mean pairwise MMD$^2$",
                            "SAM: loss at theta (solid)\nand at theta+eps (dotted)", "mean source-val macro-F1"]):
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("epoch")
handles = [Line2D([], [], color=RUN_COLORS[r], marker=RUN_MARKERS[r], markersize=4, label=RUN_LABELS[r]) for r in RUNS]
fig.legend(handles=handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.12), fontsize=7, columnspacing=1.0)
fig.tight_layout(w_pad=0.6)
save_fig(fig, "fig_training_curves")

In [ ]:
# per-class change on Sketch against ERM, with the task 2 DAN for comparison (hatched)
fig, ax = plt.subplots(figsize=(7.0, 2.4))
width = 0.26
bars_spec = [("DAN (task 2)", "dan_task2", "///"), ("DAN-DG", "dan_dg", None), ("SAM", "sam", None)]
for j, (col, key, hatch) in enumerate(bars_spec):
    vals = 100 * PER_CLASS[f"{col} change"].values
    bars = ax.bar(np.arange(N_CLASSES) + (j - 1) * width, vals, width, color=RUN_COLORS[key], edgecolor="white",
                  linewidth=0.6, hatch=hatch, label=col)
    for b, v in zip(bars, vals):
        ax.annotate(f"{v:+.0f}", xy=(b.get_x() + b.get_width() / 2, v), xytext=(0, 2 if v >= 0 else -2),
                    textcoords="offset points", ha="center", va="bottom" if v >= 0 else "top", fontsize=6, color=INK)
ax.axhline(0, color=INK, linewidth=0.8)
ax.set_xticks(np.arange(N_CLASSES))
ax.set_xticklabels([f"{n}\n{100 * a:.0f}%" for n, a in zip(CLASS_NAMES, PER_CLASS["ERM"])], fontsize=8)
ax.set_xlabel("Sketch class (ERM accuracy underneath)")
ax.set_ylabel("accuracy change vs ERM (pp)")
ax.grid(axis="x", visible=False)
ax.margins(y=0.18)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2))
save_fig(fig, "fig_per_class_sketch_change")

In [ ]:
# do the source-side diagnostics predict Sketch? separability, sharpness and worst-source F1 against Sketch accuracy
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.3))
for run in RUNS:
    e = EVAL[run]
    for ax, x in zip(axes, [100 * e["source_separability"], e["sharpness"]["delta_sharp"], 100 * e["val"]["worst_f1"]]):
        ax.scatter(x, 100 * e["sketch_acc"], s=55, color=RUN_COLORS[run], marker=RUN_MARKERS[run], edgecolor="white",
                   linewidth=0.7, zorder=3, label=RUN_LABELS[run])
axes[0].axvline(100 / 3, color="#999999", linewidth=0.8, linestyle="--")
for ax, xl in zip(axes, ["source-domain separability (%), 33 = chance", "sharpness proxy (loss increase)", "worst-source macro-F1 (%)"]):
    ax.set_xlabel(xl, fontsize=7.5)
axes[0].set_ylabel("Sketch accuracy (%)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=7)
fig.tight_layout(w_pad=0.8)
save_fig(fig, "fig_diagnostics_vs_sketch")

In [ ]:
# controlled study: what stronger source alignment does
fig, axes = plt.subplots(1, 4, figsize=(7.0, 1.95))
lams = CFG["study_lambdas"]
study = STUDY_TABLE[STUDY_TABLE["lambda"] > 0]
for ax, col, title, pct in zip(axes, ["mean src F1", "source sep.", "sharpness", "sketch acc"],
                               ["mean source-val macro-F1 (%)", "source separability (%)", "sharpness proxy", "Sketch accuracy (%)"],
                               [True, True, False, True]):
    s = 100 if pct else 1
    ax.plot(lams, s * study[col].values, color=RUN_COLORS["dan_dg"], marker="s", markersize=5, linewidth=1.8,
            markeredgecolor="white", label="DAN-DG")
    ax.axhline(s * STUDY_TABLE[col].values[0], color=RUN_COLORS["erm"], linestyle="--", linewidth=1.2, label="ERM")
    ax.set_xscale("log")
    ax.set_xticks(lams)
    ax.set_xticklabels([str(l) for l in lams])
    ax.minorticks_off()
    ax.set_xlabel("MMD weight lambda")
    ax.set_title(title, fontsize=8)
axes[0].legend(fontsize=7)
fig.tight_layout(w_pad=0.7)
save_fig(fig, "fig_lambda_study")

In [ ]:
# confusion matrices on Sketch, rows add up to 100%
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.4))
short = [n[:4] for n in CLASS_NAMES]
for ax, run in zip(axes, MAIN_RUNS):
    cm = np.array(EVAL[run]["confusion"], dtype=float)
    cm = 100 * cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, cmap=TEAL_CMAP, vmin=0, vmax=100)
    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            if cm[r, c] >= 10:
                ax.text(c, r, f"{cm[r, c]:.0f}", ha="center", va="center", fontsize=6, color="white" if cm[r, c] > 55 else INK)
    ax.set_title(f"{RUN_LABELS[run]} ({100 * EVAL[run]['sketch_acc']:.1f}%)", fontsize=8, color=RUN_COLORS[run], fontweight="bold")
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(short, rotation=90, fontsize=6.5)
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(short if ax is axes[0] else [], fontsize=6.5)
    ax.grid(False)
axes[0].set_ylabel("true class")
fig.supxlabel("predicted class", fontsize=8, y=-0.04)
fig.tight_layout(w_pad=0.4)
save_fig(fig, "fig_sketch_confusions")

In [ ]:
# failures: sketches of the class ERM handles worst, with every model's answer
worst = int(np.argmin(EVAL["erm"]["per_class_acc"]))
pred_base = np.array(EVAL["erm"]["sketch_pred"])
wrong = np.where((SKETCH_LABELS == worst) & (pred_base != worst))[0]
show = np.random.default_rng(CFG["seed"]).permutation(wrong)[:5]
fig, axes = plt.subplots(1, 5, figsize=(7.0, 2.3))
for ax in axes:
    ax.axis("off")
for ax, j in zip(axes, show):
    ax.imshow(SKETCH_IMAGES[j].permute(1, 2, 0).numpy())
    ax.set_title(f"true: {CLASS_NAMES[worst]}", fontsize=7.5)
    for k, run in enumerate(MAIN_RUNS):
        p = EVAL[run]["sketch_pred"][j]
        ax.text(0.0, -0.06 - 0.13 * k, f"{RUN_LABELS[run]}: {CLASS_NAMES[p]}", transform=ax.transAxes, fontsize=6.5,
                fontweight="bold", va="top", ha="left", color=GOOD if p == worst else BAD)
fig.subplots_adjust(bottom=0.38, wspace=0.08)
save_fig(fig, "fig_sketch_failures")

## Commit to GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
%cd {REPO}
!git config user.name "mardyweb"
!git config user.email "maryamw17@outlook.com"
!git remote set-url origin https://{token}@github.com/mardyweb/atml-pa1.git
!git add task3
!git commit -m "Task 3: notebook, results and figures"
!git push